# Subcluster Cards — `01b_subcluster_cards.ipynb`

Generate a **self-contained HTML report** and a **PowerPoint deck** (one slide per subcluster) from the outputs of `01_disease_subcluster_detection.ipynb`.

Run this notebook **once per tissue**, using the same `RESULTS_DIR` you used in Notebook 1.

**Outputs:**
- `{OUTPUT_DIR}/{TISSUE}_subcluster_report.html` — interactive offline HTML (DataTables, downloadable plots)
- `{OUTPUT_DIR}/{TISSUE}_subcluster_cards.pptx` — static PowerPoint deck


## 1. Imports

In [ ]:
from __future__ import annotations

import base64
import io
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from jinja2 import Environment, FileSystemLoader
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.util import Cm, Inches, Pt

from scutils.plotting.volcano_plot import volcano_plot
from scutils.plotting.functional import create_pathway_dotplot

matplotlib.use("Agg")
warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports OK.")


## 2. Configuration

Key parameters:
- `ADATA_PATH` — `.h5ad` file used in Notebook 1 (required for UMAP panels)
- `RESULTS_DIR` — output directory from Notebook 1 for this tissue
- `TISSUE` — tissue/dataset label (used in filenames and report headers)
- `SUBCLUSTER_KEY` — `adata.obs` column that holds subcluster labels (default: `"disease_subcluster"`)
- `DISEASE_KEY` / `CELLTYPE_KEY` / `CONDITION_KEY` — `adata.obs` columns
- `CONTROL_LABEL` — value of `CONDITION_KEY` that represents healthy/control
- `OUTPUT_DIR` — where HTML and PPTX are saved (defaults to `RESULTS_DIR`)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input ─────────────────────────────────────────────────────────────────────
ADATA_PATH    : Path = Path("data/my_tissue.h5ad")
RESULTS_DIR   : Path = Path("results/disease_subclusters/my_tissue")
TISSUE        : str  = "my_tissue"

# ── AnnData keys ──────────────────────────────────────────────────────────────
SUBCLUSTER_KEY : str = "disease_subcluster"
CELLTYPE_KEY   : str = "cell_type"
DISEASE_KEY    : str = "disease"          # column with disease/condition labels
CONDITION_KEY  : str = "disease"          # column used to split healthy vs disease
CONTROL_LABEL  : str = "control"          # value in CONDITION_KEY for healthy cells

# ── DE / enrichment filenames ─────────────────────────────────────────────────
# Pattern for DE results CSV (relative to RESULTS_DIR/de/).
# {label} will be substituted with the subcluster label (spaces → underscores).
DE_FILENAME_PATTERN         : str = "{label}_DE_results.csv"
ENRICHMENT_FILENAME_PATTERN : str = "{label}_enrichment.csv"

# DE columns expected in the CSV
DE_PVAL_COL : str = "padj"
DE_LFC_COL  : str = "log2FoldChange"

# ── Volcano plot settings ─────────────────────────────────────────────────────
VOLCANO_PVAL_CUTOFF : float = 0.05
VOLCANO_LFC_CUTOFF  : float = 0.5
VOLCANO_TOP_N_UP    : int   = 10
VOLCANO_TOP_N_DOWN  : int   = 5
VOLCANO_FIGSIZE     : tuple = (10, 6)

# ── Enrichment dotplot settings ───────────────────────────────────────────────
ENRICHMENT_MAX_PATHWAYS : int   = 15
ENRICHMENT_FIGSIZE      : tuple = (12, 6)
PATHWAY_COLORS          : dict  = {
    "GO:BP": "#1f77b4",
    "GO:MF": "#ff7f0e",
    "GO:CC": "#2ca02c",
    "KEGG":  "#d62728",
    "REAC":  "#9467bd",
    "WP":    "#CD853F",
}

# ── UMAP settings ─────────────────────────────────────────────────────────────
UMAP_FIGSIZE        : tuple = (6, 5)
UMAP_POINT_SIZE     : float = None   # None = auto
BACKGROUND_COLOR    : str   = "#d3d3d3"

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR   : Path = RESULTS_DIR   # override if needed
SAVE_FIGS    : bool = True          # also save individual PNGs to OUTPUT_DIR/figures/

# ── PPTX layout ───────────────────────────────────────────────────────────────
PPTX_TOP_DE_GENES   : int = 20   # rows in the DE gene table on each slide

# ─────────────────────────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"
if SAVE_FIGS:
    FIG_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"  Tissue      : {TISSUE}")
print(f"  Results dir : {RESULTS_DIR}")
print(f"  Output dir  : {OUTPUT_DIR}")


## 3. Load Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(f"Loaded: {adata}")

# Validate required keys
for key, attr in [
    (SUBCLUSTER_KEY, "obs"),
    (CELLTYPE_KEY,   "obs"),
    (DISEASE_KEY,    "obs"),
]:
    assert key in getattr(adata, attr).columns, (
        f"Key '{key}' not found in adata.{attr}. "
        f"Available: {list(getattr(adata, attr).columns)}"
    )

info_key = f"{SUBCLUSTER_KEY}_info"
assert info_key in adata.uns, (
    f"'{info_key}' not found in adata.uns. "
    "Ensure you ran 01_disease_subcluster_detection.ipynb first."
)

subcluster_info: pd.DataFrame = adata.uns[info_key].copy()
background_mask = adata.obs[SUBCLUSTER_KEY] == "background"

# All non-background subcluster labels
subcluster_labels = [
    lbl for lbl in adata.obs[SUBCLUSTER_KEY].unique()
    if lbl != "background"
]
print(f"Found {len(subcluster_labels)} disease-enriched subclusters:")
for lbl in sorted(subcluster_labels):
    print(f"  {lbl}")


## 4. Helper Functions

In [ ]:
def _fig_to_b64(fig: plt.Figure) -> str:
    """Encode a matplotlib Figure as a base64 PNG string."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


def _fig_to_bytes(fig: plt.Figure) -> bytes:
    """Return a matplotlib Figure as PNG bytes."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=150)
    buf.seek(0)
    return buf.read()


def _safe_label(label: str) -> str:
    """Convert a subcluster label to a filesystem-safe string."""
    return label.replace("/", "_").replace(" ", "_").replace("|", "_")


def _make_umap_celltype(adata: AnnData, figsize: tuple) -> plt.Figure:
    """UMAP coloured by cell type."""
    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.umap(adata, color=CELLTYPE_KEY, ax=ax, show=False,
               frameon=False, title="Cell type")
    return fig


def _make_umap_disease(adata: AnnData, figsize: tuple) -> plt.Figure:
    """UMAP coloured by disease condition."""
    fig, ax = plt.subplots(figsize=figsize)
    sc.pl.umap(adata, color=DISEASE_KEY, ax=ax, show=False,
               frameon=False, title="Disease")
    return fig


def _make_umap_highlight(
    adata: AnnData,
    subcluster_label: str,
    figsize: tuple,
) -> plt.Figure:
    """UMAP with subcluster cells highlighted; background cells in grey."""
    mask = adata.obs[SUBCLUSTER_KEY] == subcluster_label

    fig, ax = plt.subplots(figsize=figsize)
    coords = adata.obsm["X_umap"]
    pt_size = UMAP_POINT_SIZE or max(120000 / adata.n_obs, 0.5)
    ax.scatter(
        coords[~mask, 0], coords[~mask, 1],
        s=pt_size, c=BACKGROUND_COLOR, linewidths=0, rasterized=True,
    )
    ax.scatter(
        coords[mask, 0], coords[mask, 1],
        s=pt_size * 1.5, c="#e63946", linewidths=0, rasterized=True, zorder=2,
    )
    ax.set_title(f"Subcluster: {subcluster_label}", fontsize=10)
    ax.axis("off")
    return fig


def _fmt_float(v, decimals: int = 3) -> str:
    try:
        return f"{float(v):.{decimals}g}"
    except (ValueError, TypeError):
        return str(v)


def _parse_stats(label: str, info_df: pd.DataFrame) -> dict:
    """Extract per-subcluster stats using the actual subcluster_info column names."""
    # Match on the full label column produced by detect_disease_enriched_subclusters
    row = info_df[info_df["label"] == label]
    if row.empty:
        return {}
    row = row.iloc[0]

    n_total   = int(row["n_cells"])         if pd.notna(row.get("n_cells"))         else "—"
    n_disease = int(row["n_disease_cells"]) if pd.notna(row.get("n_disease_cells")) else "—"
    n_healthy = (
        n_total - n_disease
        if isinstance(n_total, int) and isinstance(n_disease, int)
        else "—"
    )

    return {
        "n_total":         n_total,
        "n_disease":       n_disease,
        "n_healthy":       n_healthy,
        "fold_enrichment": _fmt_float(row.get("fold_enrichment", "—")),
        "fdr_pval":        _fmt_float(row.get("pvalue_adj", "—"), decimals=2),
        "cell_type":       str(row.get("subset", "—")),
        "disease":         str(row.get("disease", "—")),
        "reference_used":  str(row.get("reference_used", "—")),
    }


def _parse_disease_breakdown(
    label: str,
    info_df: pd.DataFrame,
    adata: AnnData,
) -> list[dict]:
    """
    Build a per-disease breakdown for a subcluster.

    When combine_diseases is not None, `diseases_contributing` contains a
    comma-separated string of actual disease labels. Per-disease cell counts
    are computed directly from adata.obs so we don't need extra columns.
    When combine_diseases is None, the subcluster is already disease-specific
    (disease column = actual disease name) — return a single-row breakdown.
    """
    row = info_df[info_df["label"] == label]
    if row.empty:
        return []
    row = row.iloc[0]

    subcluster_mask = adata.obs[SUBCLUSTER_KEY] == label
    n_subcluster = subcluster_mask.sum()
    if n_subcluster == 0:
        return []

    # Case 1: diseases_contributing column present and populated
    diseases_str = row.get("diseases_contributing", "") if "diseases_contributing" in info_df.columns else ""
    if diseases_str and str(diseases_str).strip():
        diseases = [d.strip() for d in str(diseases_str).split(",") if d.strip()]
        breakdown = []
        for d in sorted(diseases):
            disease_cells_mask = (
                subcluster_mask & (adata.obs[DISEASE_KEY] == d)
            )
            n = int(disease_cells_mask.sum())
            pct = f"{100 * n / n_subcluster:.1f}%"
            breakdown.append({
                "disease": d,
                "n_cells": n,
                "pct":     pct,
                "fold":    "—",   # per-disease fold not stored separately
                "pval":    "—",
                "fdr":     "—",
            })
        return breakdown

    # Case 2: single-disease subcluster (combine_diseases=None / "union")
    disease_label = str(row.get("disease", ""))
    if disease_label and disease_label != "combined":
        n = int(row.get("n_disease_cells", 0))
        pct = f"{100 * n / n_subcluster:.1f}%" if n_subcluster > 0 else "—"
        return [{
            "disease": disease_label,
            "n_cells": n,
            "pct":     pct,
            "fold":    _fmt_float(row.get("fold_enrichment", "—")),
            "pval":    _fmt_float(row.get("pvalue", "—")),
            "fdr":     _fmt_float(row.get("pvalue_adj", "—"), decimals=2),
        }]

    return []


def _load_de(label: str) -> pd.DataFrame | None:
    """
    Load DE results CSV for a subcluster.

    Notebook 1 saves comparisons as:
        de/{safe_comparison_label}.csv
    where the comparison label starts with the safe subcluster label
    (e.g. de_Principal_cell_combined_sub1_vs_Principal_cell_Control.csv).

    We glob for any file whose name contains the safe subcluster label
    and merge results if multiple comparisons exist.
    """
    safe = _safe_label(label)
    de_dir = RESULTS_DIR / "de"
    if not de_dir.exists():
        print(f"  [warn] DE directory not found: {de_dir}")
        return None

    matches = [
        p for p in sorted(de_dir.glob(f"de_{safe}*.csv"))
        if "all_comparisons" not in p.name
    ]
    if not matches:
        # Broader search: label appears anywhere in filename
        matches = [
            p for p in sorted(de_dir.glob("de_*.csv"))
            if safe in p.stem and "all_comparisons" not in p.name
        ]
    if not matches:
        print(f"  [warn] No DE file found for '{label}' in {de_dir}")
        return None

    frames = []
    for p in matches:
        df = pd.read_csv(p)
        df["_comparison"] = p.stem.removeprefix("de_")
        frames.append(df)

    if len(frames) == 1:
        return frames[0].drop(columns=["_comparison"])
    # Multiple comparisons: keep the comparison column for context
    combined = pd.concat(frames, ignore_index=True)
    # Reorder so _comparison is first
    cols = ["_comparison"] + [c for c in combined.columns if c != "_comparison"]
    return combined[cols]


def _load_enrichment(label: str) -> pd.DataFrame | None:
    """
    Load enrichment CSV for a subcluster.

    Notebook 1 saves enrichment as:
        enrichment/enrichment_{safe_comparison_label}.csv
    We glob for any file whose name contains the safe subcluster label.
    """
    safe = _safe_label(label)
    enrich_dir = RESULTS_DIR / "enrichment"
    if not enrich_dir.exists():
        print(f"  [warn] Enrichment directory not found: {enrich_dir}")
        return None

    matches = [
        p for p in sorted(enrich_dir.glob(f"enrichment_{safe}*.csv"))
        if "all_comparisons" not in p.name
    ]
    if not matches:
        matches = [
            p for p in sorted(enrich_dir.glob("enrichment_*.csv"))
            if safe in p.stem and "all_comparisons" not in p.name
        ]
    if not matches:
        print(f"  [warn] No enrichment file found for '{label}' in {enrich_dir}")
        return None

    frames = [pd.read_csv(p) for p in matches]
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]


def _round_df(df: pd.DataFrame, decimals: int = 4) -> pd.DataFrame:
    """Round float columns for display."""
    out = df.copy()
    for col in out.select_dtypes(include="float").columns:
        out[col] = out[col].round(decimals)
    return out


print("Helper functions defined.")


## 5. Generate Per-Subcluster Assets

In [ ]:
subcluster_data = []  # list of dicts, one per subcluster

for label in sorted(subcluster_labels):
    safe = _safe_label(label)
    print(f"\n── {label}")

    card = {
        "label":             label,
        "anchor":            safe,
        "disease":           label.split("|")[1] if "|" in label else label,
        "stats":             _parse_stats(label, subcluster_info),
        "disease_breakdown": _parse_disease_breakdown(label, subcluster_info, adata),
    }

    # ── UMAP panels ──────────────────────────────────────────────────────
    print("  generating UMAP panels...", end=" ")
    try:
        fig_ct  = _make_umap_celltype(adata, UMAP_FIGSIZE)
        fig_dis = _make_umap_disease(adata, UMAP_FIGSIZE)
        fig_hl  = _make_umap_highlight(adata, label, UMAP_FIGSIZE)

        card["umap_celltype_b64"]  = _fig_to_b64(fig_ct)
        card["umap_disease_b64"]   = _fig_to_b64(fig_dis)
        card["umap_highlight_b64"] = _fig_to_b64(fig_hl)

        card["umap_celltype_bytes"]  = _fig_to_bytes(fig_ct)
        card["umap_disease_bytes"]   = _fig_to_bytes(fig_dis)
        card["umap_highlight_bytes"] = _fig_to_bytes(fig_hl)

        if SAVE_FIGS:
            fig_ct.savefig(FIG_DIR / f"{safe}_umap_celltype.png",  bbox_inches="tight", dpi=150)
            fig_dis.savefig(FIG_DIR / f"{safe}_umap_disease.png",  bbox_inches="tight", dpi=150)
            fig_hl.savefig(FIG_DIR / f"{safe}_umap_highlight.png", bbox_inches="tight", dpi=150)

        plt.close(fig_ct); plt.close(fig_dis); plt.close(fig_hl)
        print("OK")
    except Exception as e:
        print(f"FAILED ({e})")
        card.update({"umap_celltype_b64": None, "umap_disease_b64": None, "umap_highlight_b64": None,
                     "umap_celltype_bytes": None, "umap_disease_bytes": None, "umap_highlight_bytes": None})

    # ── Volcano plot ──────────────────────────────────────────────────────
    print("  generating volcano plot...", end=" ")
    de_df = _load_de(label)
    if de_df is not None:
        try:
            fig_vol = volcano_plot(
                df=de_df,
                pval_col=DE_PVAL_COL,
                lfc_col=DE_LFC_COL,
                pval_cutoff=VOLCANO_PVAL_CUTOFF,
                lfc_cutoff=VOLCANO_LFC_CUTOFF,
                top_n_up=VOLCANO_TOP_N_UP,
                top_n_down=VOLCANO_TOP_N_DOWN,
                figsize=VOLCANO_FIGSIZE,
            )
            card["volcano_b64"]   = _fig_to_b64(fig_vol)
            card["volcano_bytes"] = _fig_to_bytes(fig_vol)
            if SAVE_FIGS:
                fig_vol.savefig(FIG_DIR / f"{safe}_volcano.png", bbox_inches="tight", dpi=150)
            plt.close(fig_vol)
            print("OK")
        except Exception as e:
            print(f"FAILED ({e})")
            card["volcano_b64"] = None; card["volcano_bytes"] = None
    else:
        card["volcano_b64"] = None; card["volcano_bytes"] = None

    # ── DE table ──────────────────────────────────────────────────────────
    if de_df is not None:
        de_display = _round_df(de_df)
        card["de_columns"] = list(de_display.columns)
        card["de_records"] = de_display.values.tolist()
        card["de_df"]      = de_df
    else:
        card["de_columns"] = []; card["de_records"] = []; card["de_df"] = None

    # ── Enrichment dotplot ────────────────────────────────────────────────
    print("  generating enrichment dotplot...", end=" ")
    enrich_df = _load_enrichment(label)
    if enrich_df is not None:
        try:
            fig_dot = create_pathway_dotplot(
                data=enrich_df,
                source_colors=PATHWAY_COLORS,
                figsize=ENRICHMENT_FIGSIZE,
                max_pathways=ENRICHMENT_MAX_PATHWAYS,
                title=f"Functional enrichment — {label}",
            )
            card["dotplot_b64"]   = _fig_to_b64(fig_dot)
            card["dotplot_bytes"] = _fig_to_bytes(fig_dot)
            if SAVE_FIGS:
                fig_dot.savefig(FIG_DIR / f"{safe}_enrichment.png", bbox_inches="tight", dpi=150)
            plt.close(fig_dot)
            print("OK")
        except Exception as e:
            print(f"FAILED ({e})")
            card["dotplot_b64"] = None; card["dotplot_bytes"] = None
    else:
        card["dotplot_b64"] = None; card["dotplot_bytes"] = None

    subcluster_data.append(card)

print(f"\n✓ Assets generated for {len(subcluster_data)} subclusters.")


## 6. Render HTML Report

In [ ]:
TEMPLATE_DIR = Path(__file__).parent / "templates" if "__file__" in dir() else                Path("notebooks/disease_subclusters/templates")

# Load DataTables assets
_static = TEMPLATE_DIR / "static"
datatables_js  = (_static / "dataTables.min.js").read_text(encoding="utf-8")
datatables_css = (_static / "dataTables.min.css").read_text(encoding="utf-8")

# Render via Jinja2
env      = Environment(loader=FileSystemLoader(str(TEMPLATE_DIR)), autoescape=False)
template = env.get_template("subcluster_report.html.j2")

html_out = template.render(
    tissue=TISSUE,
    subclusters=subcluster_data,
    generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
    datatables_js=datatables_js,
    datatables_css=datatables_css,
)

html_path = OUTPUT_DIR / f"{TISSUE}_subcluster_report.html"
html_path.write_text(html_out, encoding="utf-8")
print(f"✓ HTML report written: {html_path}  ({html_path.stat().st_size / 1024:.0f} KB)")


## 7. Render PowerPoint Deck

In [ ]:
# ── Layout constants ─────────────────────────────────────────────────────────
SLIDE_W = Inches(13.33)
SLIDE_H = Inches(7.5)

_DARK   = RGBColor(0x1a, 0x1a, 0x2e)
_BLUE   = RGBColor(0x4a, 0x90, 0xd9)
_WHITE  = RGBColor(0xff, 0xff, 0xff)
_LGREY  = RGBColor(0xf0, 0xf2, 0xf5)
_TEXT   = RGBColor(0x1a, 0x1a, 0x2e)


def _add_text_box(slide, left, top, width, height, text, *, bold=False,
                  fontsize=10, color=_TEXT, align=PP_ALIGN.LEFT, wrap=True):
    txBox = slide.shapes.add_textbox(left, top, width, height)
    tf    = txBox.text_frame
    tf.word_wrap = wrap
    p      = tf.paragraphs[0]
    p.text = text
    p.alignment = align
    run = p.runs[0] if p.runs else p.add_run()
    run.font.bold = bold
    run.font.size = Pt(fontsize)
    run.font.color.rgb = color
    return txBox


def _add_image_bytes(slide, img_bytes, left, top, width, height):
    if img_bytes is None:
        return
    buf = io.BytesIO(img_bytes)
    slide.shapes.add_picture(buf, left, top, width=width, height=height)


def _add_title_bar(slide, label, disease):
    bar = slide.shapes.add_shape(
        1,  # MSO_SHAPE_TYPE.RECTANGLE
        0, 0, SLIDE_W, Cm(1.8),
    )
    bar.fill.solid()
    bar.fill.fore_color.rgb = _DARK
    bar.line.fill.background()

    _add_text_box(
        slide, Cm(0.4), Cm(0.1), Cm(20), Cm(1.6),
        label, bold=True, fontsize=14, color=_WHITE,
    )
    _add_text_box(
        slide, Cm(21), Cm(0.1), Cm(12), Cm(1.6),
        f"Disease: {disease}", bold=False, fontsize=11, color=_BLUE,
        align=PP_ALIGN.RIGHT,
    )


def _add_stats_block(slide, stats, breakdown, top=Cm(2.2)):
    """Compact stats text block on the left side of the slide."""
    lines = [
        f"Cells (total / disease / healthy): "
        f"{stats.get('n_total','—')} / {stats.get('n_disease','—')} / {stats.get('n_healthy','—')}",
        f"Fold enrichment: {stats.get('fold_enrichment','—')}   FDR p-value: {stats.get('fdr_pval','—')}",
        f"Cell type: {stats.get('cell_type','—')}   Disease label: {stats.get('disease','—')}",
        f"Reference: {stats.get('reference_used','—')}",
    ]
    if breakdown:
        lines.append("Per-disease contribution:")
        for r in breakdown:
            lines.append(
                f"  {r['disease']}: n={r['n_cells']} ({r['pct']})"
                + (f"  fold={r['fold']}" if r['fold'] != '—' else "")
                + (f"  FDR={r['fdr']}"  if r['fdr']  != '—' else "")
            )
    _add_text_box(
        slide, Cm(0.4), top, Cm(13), Cm(3.5),
        "\n".join(lines), fontsize=8, color=_TEXT,
    )


def _set_cell_text(cell, text: str, *, bold: bool = False, fontsize: int = 7,
                   bg_color: RGBColor | None = None):
    """Set text and font on a pptx table cell safely (works even if runs is empty)."""
    if bg_color is not None:
        cell.fill.solid()
        cell.fill.fore_color.rgb = bg_color
    tf = cell.text_frame
    tf.word_wrap = False
    p = tf.paragraphs[0]
    p.clear()                   # remove any existing runs
    run = p.add_run()
    run.text = text
    run.font.bold = bold
    run.font.size = Pt(fontsize)


def _add_de_table(slide, de_df, top, left=Cm(0.4), width=Cm(13)):
    """Render top-N DE genes as a pptx table shape."""
    if de_df is None or de_df.empty:
        return
    show_cols = [c for c in [
        "_comparison", "gene", "geneName", "log2FoldChange", "logfoldchanges",
        "padj", "pvals_adj", "pvalue", "pvals",
    ] if c in de_df.columns][:6]
    if not show_cols:
        show_cols = list(de_df.columns[:6])
    sub = de_df[show_cols].head(PPTX_TOP_DE_GENES).fillna("").copy()
    for c in sub.select_dtypes(include="float").columns:
        sub[c] = sub[c].round(4).astype(str)

    n_rows = len(sub) + 1   # +1 header
    n_cols = len(show_cols)
    tbl    = slide.shapes.add_table(n_rows, n_cols, left, top, width, Cm(0.45 * n_rows)).table

    for ci, col in enumerate(show_cols):
        _set_cell_text(tbl.cell(0, ci), col, bold=True, bg_color=_LGREY)

    for ri, (_, row) in enumerate(sub.iterrows(), start=1):
        for ci, col in enumerate(show_cols):
            _set_cell_text(tbl.cell(ri, ci), str(row[col]))


prs = Presentation()
prs.slide_width  = SLIDE_W
prs.slide_height = SLIDE_H
blank_layout = prs.slide_layouts[6]   # blank

for card in subcluster_data:
    slide = prs.slides.add_slide(blank_layout)

    _add_title_bar(slide, card["label"], card["disease"])
    _add_stats_block(slide, card["stats"], card["disease_breakdown"])

    # ── Images ────────────────────────────────────────────────────────────
    # Row 1: 3× UMAP (top-right area)
    umap_top    = Cm(2.0)
    umap_h      = Cm(5.5)
    umap_w      = Cm(5.0)
    umap_left_0 = Cm(13.5)
    gap         = Cm(0.25)

    for i, key in enumerate(["umap_celltype_bytes", "umap_disease_bytes", "umap_highlight_bytes"]):
        _add_image_bytes(slide, card.get(key),
                         umap_left_0 + i * (umap_w + gap), umap_top, umap_w, umap_h)

    # Row 2 (bottom-right): volcano + dotplot side by side
    plot_top = Cm(7.8)
    plot_h   = Cm(6.5)
    plot_w   = Cm(7.5)
    _add_image_bytes(slide, card.get("volcano_bytes"),  umap_left_0,                plot_top, plot_w, plot_h)
    _add_image_bytes(slide, card.get("dotplot_bytes"),  umap_left_0 + plot_w + gap, plot_top, plot_w, plot_h)

    # DE table (bottom-left)
    _add_de_table(slide, card.get("de_df"), top=Cm(5.9))

pptx_path = OUTPUT_DIR / f"{TISSUE}_subcluster_cards.pptx"
prs.save(str(pptx_path))
print(f"✓ PowerPoint deck written: {pptx_path}  ({pptx_path.stat().st_size / 1024:.0f} KB)")


## Summary

| Output | Path |
|--------|------|
| HTML report | `{OUTPUT_DIR}/{TISSUE}_subcluster_report.html` |
| PowerPoint  | `{OUTPUT_DIR}/{TISSUE}_subcluster_cards.pptx` |
| Individual PNGs | `{OUTPUT_DIR}/figures/` (if `SAVE_FIGS=True`) |

**Next step:** share the HTML report and/or PPTX with collaborators for review.  
Once feedback is received, update `INCLUDED_SUBCLUSTERS` in `02_consensus_gene_list.ipynb`  
to restrict the consensus gene list to approved subclusters only.
